In [ ]:
from math import sqrt

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as se
import statsmodels.api as sm
import statsmodels.formula.api as smf


# Парные регрессии

Для чтения файлов Excel нам дополнительно нужны пакеты
- `xlrd` для старых форматов,
- `openpyxl` для новых (которые `.xlsx`).

In [ ]:
data = pd.read_excel("../data1/cps12.xlsx")

Удостоверимся, что мы действительно что-то загрузили.

In [ ]:
data.head()

Воспользуемся «методом вдумчивого взгляда» и посмотрим на диаграмму рассеяния.

Будем использовать пакет `seaborn`.
Параметр `data=` показывает, из какого набора данных брать `x` и `y`.

In [ ]:
se.scatterplot(data=data, x="age", y="ahe")

Построим корреляционную матрицу `age` и `ahe`.
Я предпочитаю оборачивать все в функцию `print` чтобы избежать переформатирования со стороны Jupyter.

In [ ]:
print(data[["ahe", "age"]].corr())

Оценим модель.
Я покажу два с половиной способа это сделать.
«Под капотом» они делают одно и то же, но снаружи второй способ **лично мне** удобнее.

Первый способ состоит в прямом использовании класса `OLS`.
Нам надо вручную создавать матрицу объясняющих переменных, добавлять в нее константу.

Также, по-умолчанию, конструктор класса не проверяет данные на пропуски; если это не то, что нужно, то передайте параметру `missing=` значение `drop` для удаления наблюдений с пропусками или `raise` для вызова исключения.

Ниже мы последовательно:
1. Создаем матрицу экзогенных переменных (то есть объясняющих) и добавляем в нее свободный член (столбец из единиц).
2. Создаем объект `OLS`, передавая в него сначала объясняемую переменную, а затем матрицу экзогенных переменных.
3. Подгоняем модель (оцениваем ее).
4. Выводим общий результат.

In [ ]:
X = data["age"]
X = sm.add_constant(X)

model = sm.OLS(data.ahe, X)
model_fitted = model.fit()
print(model_fitted.summary())

Альтернативный способ состоит в применении метода `ols` из модуля `statsmodels.formula.api`.
Данный метод принимает на вход описание модели в стиле, принятом в `R`.
Оценивание проводится аналогично.

In [ ]:
model = smf.ols("ahe ~ age", data=data).fit()
print(model.summary())

Можно использовать функции и с классом `OLS`, но нужно не создавать объект напрямую, а вызвать метод `from_formula`.

In [ ]:
model = sm.OLS.from_formula("ahe ~ age", data=data).fit()
print(model.summary())

# Множественные регрессии

In [ ]:
datam = pd.read_excel("../data1/CollegeDistance.xls")
datam.head()

Оценим парную регрессию.

In [ ]:
model = smf.ols("ed ~ dist", data=datam).fit()
print(model.summary())

Оценим множественную регрессию.

In [ ]:
model1 = smf.ols(
    "ed ~ dist + bytest + female + black + hispanic + incomehi + ownhome + dadcoll + momcoll + cue80 + stwmfg80",
    data=datam,
).fit()
print(model1.summary())

Сравним новую оценку $\beta_{dist}$ со старой при помощи t-теста.

In [ ]:
model1.t_test(f"dist = {model.params['dist']}")

Рассчитаем $SER$ обоих регрессий.

In [ ]:
sqrt(model.ssr / model.df_resid)

In [ ]:
sqrt(model1.ssr / model1.df_resid)

Найдем модельное время обучения в школе для Боба -- темнокожего мужчину из семьи с доходом выше среднего, имеющей собственный дом, мать которого имеет высшее образование.

Создадим новый датафрейм с данными Боба.

In [ ]:
df_bob = pd.DataFrame(
    {
        "female": [0],
        "black": [1],
        "hispanic": [0],
        "dist": [20],
        "bytest": [58],
        "incomehi": [1],
        "ownhome": [1],
        "dadcoll": [0],
        "momcoll": [1],
        "cue80": [7.5],
        "stwmfg80": [9.75],
    }
)

In [ ]:
df_bob

In [ ]:
bob_ed = model1.predict(df_bob)
bob_ed

Джим имеет все характеристики такие же, как у Боба, кроме расстояния до школы (40 вместо 20).
Оценку числа лет обучения для Джима можно оценить как ${ED}_{Bob} + \beta_{dist} \times \Delta {ED}$.

In [ ]:
bob_ed + 20 * model1.params["dist"]

In [ ]:
df_jim = pd.DataFrame(
    {
        "female": [0],
        "black": [1],
        "hispanic": [0],
        "dist": [40],
        "bytest": [58],
        "incomehi": [1],
        "ownhome": [1],
        "dadcoll": [0],
        "momcoll": [1],
        "cue80": [7.5],
        "stwmfg80": [9.75],
    }
)

In [ ]:
model1.predict(df_jim)

Ожидаемо тот же результат.

Допустим, что мы хотим протестировать гипотезу $\beta_{black} = \beta_{hispanic} = 0$.

In [ ]:
model1.f_test("black = hispanic = 0")

In [ ]:
R = np.zeros((2, 12))
R[0, 4] = 1
R[1, 5] = 1

model1.f_test(R)

Допустим, что мы хотим протестировать гипотезу $\beta_{black} = 1, \beta_{hispanic} = -1$.

In [ ]:
model1.f_test("black = 1, hispanic = -1")

In [ ]:
R = np.zeros((2, 12))
R[0, 4] = 1
R[1, 5] = 1

r = np.zeros(2)
r[0] = 1
r[1] = -1

model1.f_test((R, r))

# Фиктивные регрессоры

In [ ]:
model = sm.OLS.from_formula("ahe ~ female", data=data).fit()
print(model.summary())

In [ ]:
data.loc[data.female == 0].ahe.mean().round(4)

In [ ]:
data.loc[data.female == 1].ahe.mean().round(4)

In [ ]:
model.params.sum().round(4)

# Нелинейные линеаризуемые модели

In [ ]:
model0 = sm.OLS.from_formula("ahe ~ age + female + bachelor", data=data).fit()
print(model0.summary())

In [ ]:
model1 = sm.OLS.from_formula("np.log(ahe) ~ age + female + bachelor", data=data).fit()
print(model1.summary())

Сравним эффекты увеличения возраста на 1 год в 25 лет и 33 года.

In [ ]:
nd = pd.DataFrame({"age": [25, 26, 33, 34], "female": 0, "bachelor": 0})
ahes = np.exp(model1.predict(nd))
ahes[1] - ahes[0], ahes[3] - ahes[2]

In [ ]:
model2 = sm.OLS.from_formula(
    "np.log(ahe) ~ np.log(age) + female + bachelor", data=data
).fit()
print(model2.summary())

In [ ]:
ahes = np.exp(model2.predict(nd))
ahes[1] - ahes[0], ahes[3] - ahes[2]

In [ ]:
model3 = sm.OLS.from_formula(
    "np.log(ahe) ~ age + I(age ** 2) + female + bachelor", data=data
).fit()
print(model3.summary())

In [ ]:
ahes = np.exp(model3.predict(nd))
ahes[1] - ahes[0], ahes[3] - ahes[2]

In [ ]:
model1.rsquared_adj, model2.rsquared_adj, model3.rsquared_adj

In [ ]:
model1.bic, model2.bic, model3.bic

In [ ]:
model1.aic, model2.aic, model3.aic

In [ ]:
age = np.linspace(18, 65, 1000)
nd2 = pd.DataFrame({"age": age, "female": 0, "bachelor": 1})
y1 = model1.predict(nd2)
y2 = model2.predict(nd2)
y3 = model3.predict(nd2)

In [ ]:
plt.plot(age, y1, label="log-lin")
plt.plot(age, y2, label="log-log")
plt.plot(age, y3, label="log-poly")
plt.legend()

In [ ]:
plt.plot(age, np.exp(y1), label="log-lin")
plt.plot(age, np.exp(y2), label="log-log")
plt.plot(age, np.exp(y3), label="log-poly")
plt.legend()

In [ ]:
model4 = sm.OLS.from_formula(
    "np.log(ahe) ~ age + I(age ** 2) + female + bachelor + female:bachelor", data=data
).fit()
print(model4.summary())

In [ ]:
model4 = sm.OLS.from_formula(
    "np.log(ahe) ~ age + I(age ** 2) + female*bachelor", data=data
).fit()
print(model4.summary())

In [ ]:
model1.bic, model2.bic, model3.bic, model4.bic

In [ ]:
model5 = sm.OLS.from_formula(
    "np.log(ahe) ~ (female + bachelor + age)**3", data=data
).fit()
print(model5.summary())

In [ ]:
model5.f_test("female=female:bachelor=female:age=bachelor:age=female:bachelor:age=0")

In [ ]:
model6 = sm.OLS.from_formula(
    "np.log(ahe) ~ bachelor + female * age - female", data=data
).fit()
print(model6.summary())

In [ ]:
model7 = sm.OLS.from_formula("np.log(ahe) ~ female + bachelor * age", data=data).fit()
print(model7.summary())

In [ ]:
for i, _model in enumerate((model1, model2, model3, model4, model5, model6, model7)):
    print(f"Model {i + 1} BIC: {_model.bic}")
